# EdgePCB-QC — GPU Training & Multi-Domain Evaluation Notebook
**Bharat AI-SoC Challenge 2026–27 | PS5**  

This turnkey notebook trains and evaluates two industrial PCB defect detection models for 50 epochs on GPU (Google Colab / Kaggle):
1. **Baseline Model:** Trained exclusively on DeepPCB monochrome scans (1,120 train images).
2. **Unified Multi-Domain Model:** Trained on unified multi-domain data (DeepPCB + DsPCBSD+ + native 640x640 tiled PCB-Defect V2 color images: 9,307 train images).

Both models are evaluated on:
- Untouched DeepPCB Test Split (194 unseen images, 1,143 defect instances)
- DsPCBSD+ Validation Split (1,444 color industrial AOI patches)
- Untouched External Color Test Split (390 native 640x640 crops from unseen boards 68, 80, 81, 82)

Metrics computed: TP, FP, FN at IoU $\ge$ 0.50, Per-Class Precision, Recall, F1, mAP@50, mAP@50-95, and Short-Circuit Recall Analysis.

In [ ]:
# 1. Environment & GPU Verification
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
    print(f"VRAM (GB):       {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f}")
else:
    print("[!] GPU not detected. Please enable GPU accelerator in Runtime -> Change runtime type.")

In [ ]:
# 2. Install Required Dependencies
!pip install --upgrade ultralytics opencv-python pyyaml onnx onnxruntime plotly matplotlib

In [ ]:
# 3. Clone Repository or Mount Drive
import os
from pathlib import Path

# Set workspace path (adjust if using mounted Google Drive)
WORKSPACE = Path(os.getcwd())
print(f"Active working directory: {WORKSPACE}")

In [ ]:
# 4. Prepare Unified Multi-Domain Dataset
# If running in Colab from cloned repo:
!python training/prepare_unified_dataset.py

## 5. Train Model 1 — DeepPCB Baseline (50 Epochs)
Trained exclusively on DeepPCB monochrome binarized scans to establish the baseline domain-shift vulnerability.

In [ ]:
from ultralytics import YOLO

# Train Baseline Model for 50 Epochs
baseline_model = YOLO("yolov8n.pt")
baseline_results = baseline_model.train(
    data="datasets/processed/deeppcb_yolo/data.yaml",
    epochs=50,
    batch=32,
    imgsz=640,
    device=0 if torch.cuda.is_available() else "cpu",
    project="runs/train",
    name="pcb_baseline_deeppcb_50epochs",
    patience=10,
    save=True,
    plots=True
)

## 6. Train Model 2 — Unified Multi-Domain Model (50 Epochs)
Trained on 9,307 multi-domain images (monochrome + industrial color patches + native 640x640 tiled crops).

In [ ]:
# Train Multi-Domain Model for 50 Epochs
multi_model = YOLO("yolov8n.pt")
multi_results = multi_model.train(
    data="datasets/processed/unified_yolo/data.yaml",
    epochs=50,
    batch=32,
    imgsz=640,
    device=0 if torch.cuda.is_available() else "cpu",
    project="runs/train",
    name="pcb_unified_multidomain_50epochs",
    hsv_h=0.02,
    hsv_s=0.60,
    hsv_v=0.40,
    fliplr=0.5,
    flipud=0.5,
    mosaic=1.0,
    patience=10,
    save=True,
    plots=True
)

## 7. Comparative Evaluation Across All Splits
Evaluates both 50-epoch checkpoints on:
1. DeepPCB untouched test split (194 images)
2. DsPCBSD+ validation split (1,444 images)
3. PCB-Defect V2 untouched external color test split (390 images)

In [ ]:
!python scripts/evaluate_comparative_models.py

## 8. Export Artifacts
Exports ONNX for Hailo compilation and packages training weights.

In [ ]:
# Export Multi-Domain best.pt to ONNX
best_multi_pt = "runs/train/pcb_unified_multidomain_50epochs/weights/best.pt"
if Path(best_multi_pt).exists():
    m = YOLO(best_multi_pt)
    m.export(format="onnx", imgsz=640, opset=17)
    print(f"[+] Exported ONNX to runs/train/pcb_unified_multidomain_50epochs/weights/best.onnx")